# 10年定着予測 - AutoGluon プロトタイプ

**目的**: 手動でCatBoost/LightGBM/XGBoostをアンサンブルしてきた`11_`/`12_`に対し、
AutoML（AutoGluon Tabular）を採用した場合の実力を試作・比較する。

## 12_の特徴量エンジニアリングをどこまで使うか

12_で作った特徴量は大きく2種類に分けられる。

| 種別 | 具体例 | 本ノートブックでの扱い |
|---|---|---|
| **構造的に必須な特徴量**（月次パネル→1行/社員への集約、外部知識の注入） | 月次集約統計、部署IDのTarget Encoding、欠勤パターン、360度評価タイミング、上司チーム規模、給与の相対化（等級内偏差等）、職種別乖離、クラスター特徴量 | **流用する**。AutoGluonは1行1ラベルの単一テーブルしか扱えず、24ヶ月分のパネルデータを社員1行に集約する処理や、KFold Target Encoding・分布シフト対策の相対化はAutoGluon自身では代替できないため。 |
| **GBDT/線形モデルの手助けのための機械的な工夫** | LabelEncoding、多項式特徴量（2乗項）、ビン化、複合カテゴリ交互作用の文字列結合 | **使わない**。AutoGluonは自前の前処理（カテゴリ変数の自動エンコード、木モデルによる自動的な非線形・交互作用の学習）を持つため、これらを手動で作るとAutoMLに任せる本来の狙いから外れる上、無駄な列を増やして学習を遅くする可能性がある。 |

## CV設計について

AutoGluonの`fit()`はデフォルトでランダムK-foldのバギング＋スタッキングを行うが、
EDAレポートで確認された「Train/Testで入社時期が異なり給与分布にシフトがある」という知見を踏まえ、
**`入社日`で時系列ソートした上でTrainの直近20%を`tuning_data`（検証用ホールドアウト）として明示的に渡し、
内部バギング（`num_bag_folds=0`）は無効化する**。これは11_/12_で採用したTimeSeriesSplitの思想に合わせたもの。
（本格運用する場合は、複数の時系列ホールドアウトを組み合わせたより頑健な検証も検討する）

## 実行環境
Google Colab（GPU: T4、ハイメモリ）を想定。AutoGluonインストール後、依存パッケージのバージョン更新により
**ランタイムの再起動が必要になる場合がある**（Colabでよくある挙動）。

In [1]:
# AutoGluon (Tabular, NN系モデルを含むフル版) のインストール
# インストール後にランタイムが自動 or 手動で再起動を促す場合があります。
# その場合は再起動後、このセルより下から再実行してください。
!pip install -q autogluon.tabular[all]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 1.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.1/108.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 23.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 37.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 MB 26.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━

In [2]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Aug  8 14:40:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             30W /   70W |     129MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import sys
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

# プロジェクトルートの設定（Google Drive）
PROJECT_ROOT = Path("/content/drive/MyDrive/jaggle_2026")
sys.path.append(str(PROJECT_ROOT))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import datetime
import warnings

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from autogluon.tabular import TabularPredictor

from common.utils.logger import get_logger
from common.utils.metrics import calculate_logloss
from common.utils.seed import seed_everything

warnings.filterwarnings("ignore")

SEED = 42
seed_everything(seed=SEED)

TARGET_COL = "10年定着ラベル"
ID_COL = "社員ID"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [5]:
# スクリプト名・日付・保存パスの設定
SCRIPT_NAME = "13_autogluon_prototype"
TODAY = datetime.datetime.now().strftime("%Y%m%d")

LOG_DIR = PROJECT_ROOT / "logs"
logger = get_logger(SCRIPT_NAME, log_dir=str(LOG_DIR))
logger.info(f"=== [{SCRIPT_NAME}] 実験開始 ===")

OUTPUT_DIR = PROJECT_ROOT / "data" / "output" / TODAY
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_autogluon.csv"

SAVED_MODELS_DIR = PROJECT_ROOT / "saved_models" / TODAY / SCRIPT_NAME
SAVED_MODELS_DIR.mkdir(parents=True, exist_ok=True)
AUTOGLUON_MODEL_DIR = SAVED_MODELS_DIR / "autogluon"

logger.info(f"Output Directory: {OUTPUT_DIR}")
logger.info(f"AutoGluon Model Directory: {AUTOGLUON_MODEL_DIR}")

[2026-08-08 14:40:26] [INFO] === [13_autogluon_prototype] 実験開始 ===


INFO:13_autogluon_prototype:=== [13_autogluon_prototype] 実験開始 ===


[2026-08-08 14:40:26] [INFO] Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260808


INFO:13_autogluon_prototype:Output Directory: /content/drive/MyDrive/jaggle_2026/data/output/20260808


[2026-08-08 14:40:26] [INFO] AutoGluon Model Directory: /content/drive/MyDrive/jaggle_2026/saved_models/20260808/13_autogluon_prototype/autogluon


INFO:13_autogluon_prototype:AutoGluon Model Directory: /content/drive/MyDrive/jaggle_2026/saved_models/20260808/13_autogluon_prototype/autogluon


In [6]:
# データの読み込み
INPUT_DIR = PROJECT_ROOT / "data" / "input"

train_persona = pd.read_csv(INPUT_DIR / "employee_persona_train.csv")
test_persona = pd.read_csv(INPUT_DIR / "employee_persona_test.csv")
train_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_train.csv")
test_monthly = pd.read_csv(INPUT_DIR / "employee_monthly_test.csv")

logger.info(f"Train Persona Shape: {train_persona.shape}, Test Persona Shape: {test_persona.shape}")
logger.info(f"Train Monthly Shape: {train_monthly.shape}, Test Monthly Shape: {test_monthly.shape}")

y_train = train_persona[TARGET_COL]
train_ids = train_persona[ID_COL].values
test_ids = test_persona[ID_COL].values

logger.info(f"定着率: {y_train.mean():.4f}")
logger.info(f"Train IDs: {len(train_ids)}, Test IDs: {len(test_ids)}")

[2026-08-08 14:40:26] [INFO] Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


INFO:13_autogluon_prototype:Train Persona Shape: (2761, 20), Test Persona Shape: (2502, 19)


[2026-08-08 14:40:26] [INFO] Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


INFO:13_autogluon_prototype:Train Monthly Shape: (65754, 29), Test Monthly Shape: (60048, 29)


[2026-08-08 14:40:26] [INFO] 定着率: 0.5647


INFO:13_autogluon_prototype:定着率: 0.5647


[2026-08-08 14:40:26] [INFO] Train IDs: 2761, Test IDs: 2502


INFO:13_autogluon_prototype:Train IDs: 2761, Test IDs: 2502


## 1. 特徴量エンジニアリング関数の定義（12_から流用）

月次パネルデータ（社員×24ヶ月）を社員1行に集約する関数群、および部署ID Target Encoding等の
外部知識を注入する関数群を、12_からそのまま流用する。LabelEncoding・多項式・ビン化・複合カテゴリ交互作用は
AutoGluon自身の前処理・モデル群に任せるため、ここでは実装しない。

In [7]:
def create_monthly_aggregation_features(monthly_df, employee_ids):
    """月次データから集約特徴量を生成（12_と同一ロジック）"""
    numeric_cols = [
        "残業時間", "有給取得日数", "欠勤日数", "研修時間",
        "上司との面談実施回数", "情報共有件数", "在宅勤務日数",
        "360度評価_親和度", "360度評価_信頼度", "360度評価_主体度",
        "360度評価_学習度", "360度評価_共有貢献度", "360度評価者数",
        "顧客満足度評価", "担当プロジェクト数", "月例給与_円"
    ]

    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].copy()
        emp_data = emp_data.sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        for col in numeric_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            valid_values = values[~pd.isna(values)]

            features[f"{col}_mean"] = np.mean(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_std"] = np.std(valid_values) if len(valid_values) > 1 else np.nan
            features[f"{col}_min"] = np.min(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_max"] = np.max(valid_values) if len(valid_values) > 0 else np.nan
            features[f"{col}_median"] = np.median(valid_values) if len(valid_values) > 0 else np.nan
            mean_val = features[f"{col}_mean"]
            std_val = features[f"{col}_std"]
            features[f"{col}_cv"] = std_val / mean_val if (mean_val and mean_val != 0) else np.nan

            early = emp_data[emp_data["経過月数"].between(0, 2)][col]
            mid = emp_data[emp_data["経過月数"].between(3, 11)][col]
            late = emp_data[emp_data["経過月数"].between(12, 23)][col]
            features[f"{col}_early_mean"] = early.mean()
            features[f"{col}_mid_mean"] = mid.mean()
            features[f"{col}_late_mean"] = late.mean()
            features[f"{col}_late_minus_early"] = late.mean() - early.mean()
            features[f"{col}_late_early_ratio"] = (
                late.mean() / early.mean() if early.mean() and early.mean() != 0 else np.nan
            )

            if len(valid_values) >= 2:
                valid_indices = np.where(~pd.isna(values))[0]
                if len(valid_indices) >= 2:
                    slope, _, _, _, _ = stats.linregress(valid_indices, valid_values)
                    features[f"{col}_slope"] = slope
                else:
                    features[f"{col}_slope"] = np.nan
                first_val, last_val = valid_values[0], valid_values[-1]
                features[f"{col}_diff"] = last_val - first_val
                features[f"{col}_ratio"] = last_val / first_val if first_val != 0 else np.nan
            else:
                features[f"{col}_slope"] = features[f"{col}_diff"] = features[f"{col}_ratio"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_monthly_categorical_change_features(monthly_df, employee_ids):
    categorical_cols = ["部署ID", "職種", "役割", "等級", "勤務地", "上司ID"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in categorical_cols:
            if col not in emp_data.columns:
                continue
            values = emp_data[col].values
            changes = sum(1 for i in range(1, len(values)) if pd.notna(values[i]) and pd.notna(values[i-1]) and values[i] != values[i-1])
            features[f"{col}_changes"] = changes
            features[f"{col}_unique_count"] = len(pd.Series(values).dropna().unique())
        if "月末在籍状態" in emp_data.columns:
            status_values = emp_data["月末在籍状態"].values
            features["leave_of_absence_flag"] = int("休職" in status_values)
            features["leave_of_absence_months"] = np.sum(status_values == "休職")
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_missing_value_features(monthly_df, employee_ids):
    missing_target_cols = ["360度評価_親和度", "360度評価_信頼度", "顧客満足度評価", "担当プロジェクト数"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in missing_target_cols:
            if col in emp_data.columns:
                values = emp_data[col].values
                total_months = len(values)
                features[f"{col}_missing_rate"] = pd.isna(values).sum() / total_months if total_months > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_domain_knowledge_features(monthly_df, employee_ids):
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
        eval_mean_list = [emp_data[col].mean() for col in eval_cols if col in emp_data.columns]
        features["engagement_score"] = np.nanmean(eval_mean_list) if len(eval_mean_list) > 0 else np.nan
        if "残業時間" in emp_data.columns:
            features["overtime_stability"] = emp_data["残業時間"].std()
        if "研修時間" in emp_data.columns and "残業時間" in emp_data.columns:
            training_mean = emp_data["研修時間"].mean()
            overtime_mean = emp_data["残業時間"].mean()
            features["training_overtime_ratio"] = training_mean / overtime_mean if overtime_mean > 0 else np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)

print("✅ 月次集約・カテゴリ変化・欠損値・ドメイン特徴量関数定義完了")

✅ 月次集約・カテゴリ変化・欠損値・ドメイン特徴量関数定義完了


In [8]:
def create_advanced_statistical_features(monthly_df, employee_ids):
    """統計的特徴量：歪度、尖度、パーセンタイル"""
    numeric_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}
        for col in numeric_cols:
            if col in emp_data.columns:
                values = emp_data[col].dropna().values
                if len(values) >= 3:
                    features[f"{col}_skew"] = stats.skew(values)
                    features[f"{col}_kurtosis"] = stats.kurtosis(values)
                    features[f"{col}_q25"] = np.percentile(values, 25)
                    features[f"{col}_q75"] = np.percentile(values, 75)
                    features[f"{col}_iqr"] = features[f"{col}_q75"] - features[f"{col}_q25"]
                else:
                    features[f"{col}_skew"] = features[f"{col}_kurtosis"] = np.nan
                    features[f"{col}_q25"] = features[f"{col}_q75"] = features[f"{col}_iqr"] = np.nan
        features_list.append(features)
    return pd.DataFrame(features_list)


def create_cluster_features(monthly_df, employee_ids, n_clusters=5, seed=42):
    """クラスター特徴量：月次データの平均をKMeansクラスタリング"""
    key_cols = ["残業時間", "有給取得日数", "研修時間", "360度評価_親和度", "360度評価_信頼度"]
    agg_data = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id]
        row = {"社員ID": employee_id}
        for col in key_cols:
            if col in emp_data.columns:
                row[col] = emp_data[col].mean()
        agg_data.append(row)
    agg_df = pd.DataFrame(agg_data)
    feature_cols = [c for c in key_cols if c in agg_df.columns]
    X = agg_df[feature_cols].fillna(-999)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    kmeans = KMeans(n_clusters=n_clusters, random_state=seed, n_init=10)
    agg_df["cluster"] = kmeans.fit_predict(X_scaled)
    return agg_df[["社員ID", "cluster"]]


def create_department_target_encoding(train_persona, test_persona, y_train, seed=42, n_splits=5, smoothing=10):
    """初期部署IDのKFold + スムージング付きTarget Encoding（12_と同一ロジック）"""
    col = "初期部署ID"
    global_mean = y_train.mean()

    train_te = np.zeros(len(train_persona))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    dept_train = train_persona[col].values
    y_arr = y_train.values

    for tr_idx, val_idx in kf.split(train_persona):
        df_tr = pd.DataFrame({col: dept_train[tr_idx], "y": y_arr[tr_idx]})
        stats_tr = df_tr.groupby(col)["y"].agg(["mean", "count"])
        smoothed = (stats_tr["count"] * stats_tr["mean"] + smoothing * global_mean) / (stats_tr["count"] + smoothing)
        mapping = smoothed.to_dict()
        train_te[val_idx] = pd.Series(dept_train[val_idx]).map(mapping).fillna(global_mean).values

    df_full = pd.DataFrame({col: dept_train, "y": y_arr})
    stats_full = df_full.groupby(col)["y"].agg(["mean", "count"])
    smoothed_full = (stats_full["count"] * stats_full["mean"] + smoothing * global_mean) / (stats_full["count"] + smoothing)
    mapping_full = smoothed_full.to_dict()
    dept_size_map = stats_full["count"].to_dict()

    test_te = test_persona[col].map(mapping_full).fillna(global_mean).values

    train_out = pd.DataFrame({
        "社員ID": train_persona["社員ID"].values,
        "dept_target_enc": train_te,
        "dept_size": train_persona[col].map(dept_size_map).fillna(0).values,
    })
    test_out = pd.DataFrame({
        "社員ID": test_persona["社員ID"].values,
        "dept_target_enc": test_te,
        "dept_size": test_persona[col].map(dept_size_map).fillna(0).values,
    })
    return train_out, test_out


def create_eda_driven_features(monthly_df, employee_ids):
    """欠勤日数パターン・360度評価タイミング・月次ボラティリティ・比率特徴量（12_と同一ロジック）"""
    eval_cols = ["360度評価_親和度", "360度評価_信頼度", "360度評価_主体度", "360度評価_学習度", "360度評価_共有貢献度"]
    features_list = []
    for employee_id in employee_ids:
        emp_data = monthly_df[monthly_df["社員ID"] == employee_id].sort_values("経過月数").reset_index(drop=True)
        features = {"社員ID": employee_id}

        absence_vals = emp_data["欠勤日数"].values
        nonzero = absence_vals > 0
        features["欠勤発生月数"] = int(nonzero.sum())
        max_run = cur_run = 0
        for v in nonzero:
            cur_run = cur_run + 1 if v else 0
            max_run = max(max_run, cur_run)
        features["欠勤_最長連続月数"] = max_run
        features["欠勤_連続フラグ"] = int(max_run >= 2)

        flagged = emp_data[emp_data["360度評価更新フラグ"] == 1]
        first_month = flagged["経過月数"].min() if len(flagged) > 0 else np.nan
        features["初回評価月"] = first_month
        features["is_早期評価"] = int(first_month <= 4) if pd.notna(first_month) else 0
        features["is_遅延評価"] = int(first_month >= 7) if pd.notna(first_month) else 0
        features["評価遅延度"] = abs(first_month - 5) if pd.notna(first_month) else np.nan

        for col in ["残業時間", "月例給与_円"]:
            vals = emp_data[col].dropna().values
            features[f"{col}_volatility"] = np.mean(np.abs(np.diff(vals))) if len(vals) >= 2 else np.nan

        n_months = len(emp_data)
        features["有給取得率"] = emp_data["有給取得日数"].sum() / n_months if n_months > 0 else np.nan
        features["評価項目間ばらつき"] = emp_data[eval_cols].std(axis=1).mean()

        features_list.append(features)
    return pd.DataFrame(features_list)


def create_manager_team_size_features(monthly_df, employee_ids):
    """初期（経過月数=0）時点で同じ上司IDを持つ社員数（12_と同一ロジック）"""
    month0 = monthly_df[monthly_df["経過月数"] == 0].copy()
    month0["初期上司_部下数"] = month0.groupby("上司ID")["社員ID"].transform("count")
    out = month0[["社員ID", "初期上司_部下数"]]
    return out[out["社員ID"].isin(employee_ids)].reset_index(drop=True)

print("✅ 高度統計・クラスター・部署Target Encoding・EDA駆動・上司チーム規模の関数定義完了")

✅ 高度統計・クラスター・部署Target Encoding・EDA駆動・上司チーム規模の関数定義完了


In [9]:
logger.info("-" * 60)
logger.info("特徴量生成開始")
logger.info("-" * 60)

logger.info("月次集約特徴量を生成中...")
train_monthly_agg = create_monthly_aggregation_features(train_monthly, train_ids)
test_monthly_agg = create_monthly_aggregation_features(test_monthly, test_ids)

logger.info("月次カテゴリ変化特徴量を生成中...")
train_cat_change = create_monthly_categorical_change_features(train_monthly, train_ids)
test_cat_change = create_monthly_categorical_change_features(test_monthly, test_ids)

logger.info("欠損値特徴量を生成中...")
train_missing = create_missing_value_features(train_monthly, train_ids)
test_missing = create_missing_value_features(test_monthly, test_ids)

logger.info("ドメイン知識特徴量を生成中...")
train_domain = create_domain_knowledge_features(train_monthly, train_ids)
test_domain = create_domain_knowledge_features(test_monthly, test_ids)

logger.info("高度な統計特徴量を生成中...")
train_advanced_stats = create_advanced_statistical_features(train_monthly, train_ids)
test_advanced_stats = create_advanced_statistical_features(test_monthly, test_ids)

logger.info("クラスター特徴量を生成中...")
train_cluster = create_cluster_features(train_monthly, train_ids, n_clusters=5, seed=SEED)
test_cluster = create_cluster_features(test_monthly, test_ids, n_clusters=5, seed=SEED)

logger.info("部署ID Target Encodingを生成中...")
train_dept_te, test_dept_te = create_department_target_encoding(
    train_persona, test_persona, y_train, seed=SEED, n_splits=5, smoothing=10
)

logger.info("EDA駆動特徴量（欠勤パターン・評価タイミング・ボラティリティ・比率）を生成中...")
train_eda_feats = create_eda_driven_features(train_monthly, train_ids)
test_eda_feats = create_eda_driven_features(test_monthly, test_ids)

logger.info("上司チーム規模特徴量を生成中...")
train_mgr = create_manager_team_size_features(train_monthly, train_ids)
test_mgr = create_manager_team_size_features(test_monthly, test_ids)

logger.info("特徴量生成完了")

[2026-08-08 14:40:27] [INFO] ------------------------------------------------------------


INFO:13_autogluon_prototype:------------------------------------------------------------


[2026-08-08 14:40:27] [INFO] 特徴量生成開始


INFO:13_autogluon_prototype:特徴量生成開始


[2026-08-08 14:40:27] [INFO] ------------------------------------------------------------


...

INFO:13_autogluon_prototype:EDA駆動特徴量（欠勤パターン・評価タイミング・ボラティリティ・比率）を生成中...


[2026-08-08 14:47:10] [INFO] 上司チーム規模特徴量を生成中...


INFO:13_autogluon_prototype:上司チーム規模特徴量を生成中...


[2026-08-08 14:47:10] [INFO] 特徴量生成完了


INFO:13_autogluon_prototype:特徴量生成完了


## 2. Persona単位の特徴量（12_から「構造的」な部分のみ）

テキスト文字数、時間的特徴量、年齢×前職経験の交互作用、給与の相対化（分布シフト対策）、
職種別乖離（研修時間の見かけの相関という交絡への対策）を流用する。
**LabelEncoding・多項式（2乗項）・ビン化・複合カテゴリ交互作用の文字列結合は行わない**
（AutoGluonが生カテゴリ列・生の数値列から自動的にエンコード・非線形性・交互作用を学習するため）。

In [10]:
logger.info("テキスト特徴量を生成中...")
text_cols = ["入社時メモ", "上司からのフィードバック", "同僚からのフィードバック"]
train_persona["text_total_chars"] = train_persona[text_cols].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
test_persona["text_total_chars"] = test_persona[text_cols].fillna("").apply(lambda x: sum(len(str(v)) for v in x), axis=1)
for col in text_cols:
    train_persona[f"{col}_len"] = train_persona[col].fillna("").astype(str).apply(len)
    test_persona[f"{col}_len"] = test_persona[col].fillna("").astype(str).apply(len)

logger.info("時間的特徴量を生成中...")
train_persona["入社日"] = pd.to_datetime(train_persona["入社日"])
test_persona["入社日"] = pd.to_datetime(test_persona["入社日"])
train_persona["入社年"] = train_persona["入社日"].dt.year
train_persona["入社月"] = train_persona["入社日"].dt.month
train_persona["入社四半期"] = train_persona["入社日"].dt.quarter
test_persona["入社年"] = test_persona["入社日"].dt.year
test_persona["入社月"] = test_persona["入社日"].dt.month
test_persona["入社四半期"] = test_persona["入社日"].dt.quarter

logger.info("交互作用特徴量を生成中...")
train_persona["年齢_x_前職経験"] = train_persona["入社時年齢"] * train_persona["前職経験月数"]
test_persona["年齢_x_前職経験"] = test_persona["入社時年齢"] * test_persona["前職経験月数"]
grade_map = {"G1": 1, "G2": 2, "G3": 3, "G4": 4, "G5": 5}
train_persona["初期等級_num"] = train_persona["初期等級"].map(grade_map)
test_persona["初期等級_num"] = test_persona["初期等級"].map(grade_map)
train_persona["初任給_x_等級"] = train_persona["初任給_円"] * train_persona["初期等級_num"]
test_persona["初任給_x_等級"] = test_persona["初任給_円"] * test_persona["初期等級_num"]

# 入社四半期 × 入社区分（初回レポート18.2「追加3」/ 17.4節）
train_persona["is_Q2_新卒"] = ((train_persona["入社四半期"] == 2) & (train_persona["入社区分"] == "新卒")).astype(int)
test_persona["is_Q2_新卒"] = ((test_persona["入社四半期"] == 2) & (test_persona["入社区分"] == "新卒")).astype(int)

logger.info("特徴量処理完了")

[2026-08-08 14:47:10] [INFO] テキスト特徴量を生成中...


INFO:13_autogluon_prototype:テキスト特徴量を生成中...


[2026-08-08 14:47:10] [INFO] 時間的特徴量を生成中...


INFO:13_autogluon_prototype:時間的特徴量を生成中...


[2026-08-08 14:47:10] [INFO] 交互作用特徴量を生成中...


INFO:13_autogluon_prototype:交互作用特徴量を生成中...


[2026-08-08 14:47:10] [INFO] 特徴量処理完了


INFO:13_autogluon_prototype:特徴量処理完了


In [11]:
logger.info("-" * 60)
logger.info("特徴量の統合")
logger.info("-" * 60)

train_persona_features = train_persona.drop(columns=[TARGET_COL])

train_features = train_persona_features.merge(train_monthly_agg, on=ID_COL, how="left")
train_features = train_features.merge(train_cat_change, on=ID_COL, how="left")
train_features = train_features.merge(train_missing, on=ID_COL, how="left")
train_features = train_features.merge(train_domain, on=ID_COL, how="left")
train_features = train_features.merge(train_advanced_stats, on=ID_COL, how="left")
train_features = train_features.merge(train_cluster, on=ID_COL, how="left")
train_features = train_features.merge(train_dept_te, on=ID_COL, how="left")
train_features = train_features.merge(train_eda_feats, on=ID_COL, how="left")
train_features = train_features.merge(train_mgr, on=ID_COL, how="left")

test_features = test_persona.merge(test_monthly_agg, on=ID_COL, how="left")
test_features = test_features.merge(test_cat_change, on=ID_COL, how="left")
test_features = test_features.merge(test_missing, on=ID_COL, how="left")
test_features = test_features.merge(test_domain, on=ID_COL, how="left")
test_features = test_features.merge(test_advanced_stats, on=ID_COL, how="left")
test_features = test_features.merge(test_cluster, on=ID_COL, how="left")
test_features = test_features.merge(test_dept_te, on=ID_COL, how="left")
test_features = test_features.merge(test_eda_feats, on=ID_COL, how="left")
test_features = test_features.merge(test_mgr, on=ID_COL, how="left")

logger.info(f"Train: {train_features.shape}, Test: {test_features.shape}")

[2026-08-08 14:47:10] [INFO] ------------------------------------------------------------


INFO:13_autogluon_prototype:------------------------------------------------------------


[2026-08-08 14:47:10] [INFO] 特徴量の統合


INFO:13_autogluon_prototype:特徴量の統合


[2026-08-08 14:47:10] [INFO] ------------------------------------------------------------


INFO:13_autogluon_prototype:------------------------------------------------------------


[2026-08-08 14:47:10] [INFO] Train: (2761, 315), Test: (2502, 315)


INFO:13_autogluon_prototype:Train: (2761, 315), Test: (2502, 315)


In [12]:
logger.info("職種別の乖離特徴量を生成中...")
job_dev_metrics = ["残業時間_mean", "研修時間_mean", "360度評価_親和度_mean"]
job_means = {m: train_features.groupby("初期職種")[m].mean().to_dict() for m in job_dev_metrics}
category_means_train = {m: train_features.groupby("入社区分")[m].mean().to_dict() for m in job_dev_metrics}

for df in [train_features, test_features]:
    for m in job_dev_metrics:
        job_mean_series = df["初期職種"].map(job_means[m])
        df[f"{m}_job_deviation"] = df[m] - job_mean_series
    df["研修時間_職種比"] = df["研修時間_mean"] / df["初期職種"].map(job_means["研修時間_mean"]).replace(0, np.nan)
    df["研修時間_区分比"] = df["研修時間_mean"] / df["入社区分"].map(category_means_train["研修時間_mean"]).replace(0, np.nan)

logger.info("給与の相対化特徴量を生成中...")
grade_salary_mean = train_features.groupby("初期等級")["初任給_円"].mean().to_dict()
category_salary_mean = train_features.groupby("入社区分")["初任給_円"].mean().to_dict()
grade_monthly_salary_mean = train_features.groupby("初期等級")["月例給与_円_mean"].mean().to_dict()

for df in [train_features, test_features]:
    df["初任給_等級内偏差"] = df["初任給_円"] - df["初期等級"].map(grade_salary_mean)
    df["初任給_区分内偏差"] = df["初任給_円"] - df["入社区分"].map(category_salary_mean)
    df["月例給与_等級内偏差"] = df["月例給与_円_mean"] - df["初期等級"].map(grade_monthly_salary_mean)

logger.info(f"派生特徴量生成後 Train: {train_features.shape}, Test: {test_features.shape}")

[2026-08-08 14:47:10] [INFO] 職種別の乖離特徴量を生成中...


INFO:13_autogluon_prototype:職種別の乖離特徴量を生成中...


[2026-08-08 14:47:10] [INFO] 給与の相対化特徴量を生成中...


INFO:13_autogluon_prototype:給与の相対化特徴量を生成中...


[2026-08-08 14:47:10] [INFO] 派生特徴量生成後 Train: (2761, 323), Test: (2502, 323)


INFO:13_autogluon_prototype:派生特徴量生成後 Train: (2761, 323), Test: (2502, 323)


In [13]:
# 不要な列の削除
# 「初期部署ID」は Target Encoding (dept_target_enc) に情報を抽出済みのため生IDは除外
# 「最終学歴」はEDAで統計的に非有意(カイ二乗検定 p=0.221)と確認済みのため除外
# 「初期等級」は数値化(初期等級_num)済みのため生カテゴリは除外
# 生テキストは文字数特徴量に変換済みのため除外（AutoGluonのテキスト機能はここでは使わない簡易プロトタイプとする）
# ※ LabelEncodingは行わない（入社区分・性別・専攻分野・採用経路・初期職種・初期勤務地・初期役割はobject型のままAutoGluonに渡す）
drop_cols = [
    "入社時メモ", "上司からのフィードバック", "同僚からのフィードバック",
    "初期部署ID", "初期等級", "最終学歴", "前職職種",
]
drop_cols_exist = [col for col in drop_cols if col in train_features.columns]
train_features = train_features.drop(columns=drop_cols_exist)
test_features = test_features.drop(columns=drop_cols_exist)

# 社員IDはインデックスにして特徴量から除外しつつ、後で予測結果と紐付けられるようにする
train_features = train_features.set_index(ID_COL)
test_features = test_features.set_index(ID_COL)

logger.info(f"最終特徴量数: {train_features.shape[1]} (Train: {train_features.shape}, Test: {test_features.shape})")
logger.info(f"dtype内訳:\n{train_features.dtypes.value_counts()}")

[2026-08-08 14:47:10] [INFO] 最終特徴量数: 315 (Train: (2761, 315), Test: (2502, 315))


INFO:13_autogluon_prototype:最終特徴量数: 315 (Train: (2761, 315), Test: (2502, 315))


[2026-08-08 14:47:10] [INFO] dtype内訳:
float64           257
int64              46
object              7
int32               4
datetime64[ns]      1
Name: count, dtype: int64


INFO:13_autogluon_prototype:dtype内訳:
float64           257
int64              46
object              7
int32               4
datetime64[ns]      1
Name: count, dtype: int64


## 3. AutoGluonの学習・検証データ作成（時系列ホールドアウト）

EDAレポートで確認された「Train/Testで入社時期が異なり分布シフトがある」という知見を踏まえ、
`入社日`でソートしTrainの直近20%を検証用（`tuning_data`）として明示的に切り出す。
AutoGluonの内部バギング（`num_bag_folds`）は無効化し、この時系列ホールドアウトで単純に検証することで、
11_/12_のTimeSeriesSplitと同じ「過去で学習し、未来で検証する」思想を再現する。

In [14]:
target_series = train_persona.set_index(ID_COL)[TARGET_COL]

train_features_sorted = train_features.sort_values("入社日")
y_sorted = target_series.loc[train_features_sorted.index]

split_point = int(len(train_features_sorted) * 0.8)
ag_train_data = train_features_sorted.iloc[:split_point].copy()
ag_tuning_data = train_features_sorted.iloc[split_point:].copy()
ag_train_data[TARGET_COL] = y_sorted.iloc[:split_point].values
ag_tuning_data[TARGET_COL] = y_sorted.iloc[split_point:].values

logger.info(f"AutoGluon train_data: {ag_train_data.shape} (入社日 {ag_train_data['入社日'].min().date()} 〜 {ag_train_data['入社日'].max().date()})")
logger.info(f"AutoGluon tuning_data: {ag_tuning_data.shape} (入社日 {ag_tuning_data['入社日'].min().date()} 〜 {ag_tuning_data['入社日'].max().date()})")
logger.info(f"train定着率: {ag_train_data[TARGET_COL].mean():.4f}, tuning定着率: {ag_tuning_data[TARGET_COL].mean():.4f}")

[2026-08-08 14:47:11] [INFO] AutoGluon train_data: (2208, 316) (入社日 2011-04-01 〜 2013-04-01)


INFO:13_autogluon_prototype:AutoGluon train_data: (2208, 316) (入社日 2011-04-01 〜 2013-04-01)


[2026-08-08 14:47:11] [INFO] AutoGluon tuning_data: (553, 316) (入社日 2013-04-01 〜 2014-03-01)


INFO:13_autogluon_prototype:AutoGluon tuning_data: (553, 316) (入社日 2013-04-01 〜 2014-03-01)


[2026-08-08 14:47:11] [INFO] train定着率: 0.5616, tuning定着率: 0.5769


INFO:13_autogluon_prototype:train定着率: 0.5616, tuning定着率: 0.5769


## 4. AutoGluonの学習

`num_bag_folds=0, num_stack_levels=0`でAutoGluon標準の内部バギング/スタッキングを無効化し、
上で作成した時系列ホールドアウト（`tuning_data`）のみで各モデルを検証する。
`num_gpus=1`でColabのT4 GPUを指定（LightGBM/XGBoost/CatBoostはCPUで動くことが多いが、
NN系モデル（TORCH/FASTAI）はGPUを活用する）。

`presets`は指定せず、AutoGluonのデフォルト挙動（多数のモデルを学習し、検証スコアに基づいて
Greedy Weighted Ensembleを構築）に任せる。評価指標はコンペのスコアに合わせて`log_loss`を指定する。

In [15]:
predictor = TabularPredictor(
    label=TARGET_COL,
    problem_type="binary",
    eval_metric="log_loss",
    path=str(AUTOGLUON_MODEL_DIR),
)

predictor.fit(
    train_data=ag_train_data,
    tuning_data=ag_tuning_data,
    num_bag_folds=0,
    num_stack_levels=0,
    num_gpus=1,
    time_limit=3600,  # 1時間。プロトタイプのため上限を設定（必要に応じて調整）
    verbosity=2,
)

logger.info("AutoGluon学習完了")

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          8
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       48.03 GB / 50.99 GB (94.2%)
Disk Space Avail:   178.51 GB / 235.68 GB (75.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on 

[2026-08-08 14:48:12] [INFO] AutoGluon学習完了


INFO:13_autogluon_prototype:AutoGluon学習完了


## 5. 検証結果の確認

`leaderboard()`で個別モデルおよび自動構築されたWeighted Ensembleの検証スコア（log_loss）を確認する。
11_/12_のOOF Log Loss（0.585〜0.611程度）と直接比較できる。

In [18]:
leaderboard = predictor.leaderboard(ag_tuning_data, silent=True)
logger.info("=" * 60)
logger.info("AutoGluon Leaderboard (tuning_data = 時系列ホールドアウト)")
logger.info("=" * 60)
logger.info("\n" + leaderboard.to_string())

# AutoGluonのバージョンにより列名が score_test / score_val のいずれかになる場合があるため両対応
score_col = "score_test" if "score_test" in leaderboard.columns else "score_val"
best_model_name = predictor.model_best
best_score = leaderboard.loc[leaderboard["model"] == best_model_name, score_col].values[0]
# log_lossは「低いほど良い」指標だが、AutoGluon内部は符号反転して保持しているため戻す
logger.info(f"Best model: {best_model_name}, Log Loss (時系列ホールドアウト): {-best_score:.6f}")

leaderboard

[2026-08-08 14:51:09] [INFO] ============================================================


INFO:13_autogluon_prototype:============================================================


[2026-08-08 14:51:09] [INFO] AutoGluon Leaderboard (tuning_data = 時系列ホールドアウト)


INFO:13_autogluon_prototype:AutoGluon Leaderboard (tuning_data = 時系列ホールドアウト)


[2026-08-08 14:51:09] [INFO] ============================================================


...

[2026-08-08 14:51:09] [INFO] 
                  model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0   WeightedEnsemble_L2   -0.567022  -0.567022    log_loss        0.118088       0.095726  23.699342                 0.004116                0.000705           0.020038            2       True         12
1              CatBoost   -0.567941  -0.567941    log_loss        0.023682       0.021028  10.994978                 0.023682                0.021028          10.994978            1       True          5
2            LightGBMXT   -0.578975  -0.578975    log_loss        0.008906       0.006332   3.993580                 0.008906                0.006332           3.993580            1       True          1
3              LightGBM   -0.584119  -0.584119    log_loss        0.007805       0.007208   4.307941                 0.007805                0.007208     

INFO:13_autogluon_prototype:
                  model  score_test  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0   WeightedEnsemble_L2   -0.567022  -0.567022    log_loss        0.118088       0.095726  23.699342                 0.004116                0.000705           0.020038            2       True         12
1              CatBoost   -0.567941  -0.567941    log_loss        0.023682       0.021028  10.994978                 0.023682                0.021028          10.994978            1       True          5
2            LightGBMXT   -0.578975  -0.578975    log_loss        0.008906       0.006332   3.993580                 0.008906                0.006332           3.993580            1       True          1
3              LightGBM   -0.584119  -0.584119    log_loss        0.007805       0.007208   4.307941                 0.007805                0.007208      

[2026-08-08 14:51:09] [INFO] Best model: WeightedEnsemble_L2, Log Loss (時系列ホールドアウト): 0.567022


INFO:13_autogluon_prototype:Best model: WeightedEnsemble_L2, Log Loss (時系列ホールドアウト): 0.567022


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.567022,-0.567022,log_loss,0.118088,0.095726,23.699342,0.004116,0.000705,0.020038,2,True,12
1,CatBoost,-0.567941,-0.567941,log_loss,0.023682,0.021028,10.994978,0.023682,0.021028,10.994978,1,True,5
2,LightGBMXT,-0.578975,-0.578975,log_loss,0.008906,0.006332,3.993580,0.008906,0.006332,3.993580,1,True,1
3,LightGBM,-0.584119,-0.584119,log_loss,0.007805,0.007208,4.307941,0.007805,0.007208,4.307941,1,True,2
4,ExtraTreesGini,-0.587693,-0.587693,log_loss,0.134421,0.091878,0.940706,0.134421,0.091878,0.940706,1,True,6
5,XGBoost,-0.590151,-0.590151,log_loss,0.028316,0.013674,5.508142,0.028316,0.013674,5.508142,1,True,9
6,ExtraTreesEntr,-0.590178,-0.590178,log_loss,0.123162,0.091927,0.919741,0.123162,0.091927,0.919741,1,True,7
7,RandomForestGini,-0.591352,-0.591352,log_loss,0.133057,0.092633,1.800908,0.133057,0.092633,1.800908,1,True,3
8,RandomForestEntr,-0.592571,-0.592571,log_loss,0.111438,0.103988,1.817058,0.111438,0.103988,1.817058,1,True,4
9,LightGBMLarge,-0.594151,-0.594151,log_loss,0.011591,0.008458,15.077601,0.011591,0.008458,15.077601,1,True,11


## 6. Test予測と提出ファイルの作成

`predictor.predict_proba()`はデフォルトでAutoGluonが選んだ最良モデル（通常はWeighted Ensemble）を使用する。
2値分類のため、正例（定着=1）側の確率列を取り出して提出形式に整形する。

In [19]:
X_test = test_features.copy()

proba_df = predictor.predict_proba(X_test)
logger.info(f"predict_proba columns: {list(proba_df.columns)}")

# クラス列名が 1/True 等バージョンにより異なりうるため、正例列を頑健に特定する
positive_col = 1 if 1 in proba_df.columns else proba_df.columns[-1]
test_preds = proba_df[positive_col].values

sub = pd.DataFrame({ID_COL: X_test.index, TARGET_COL: test_preds})
sub.to_csv(SUBMISSION_PATH, index=False, header=False)
logger.info(f"提出ファイル保存: {SUBMISSION_PATH}")

print(f"\n■ AutoGluon 検証スコア (時系列ホールドアウト, Log Loss): {-best_score:.6f}")
print(f"■ 使用モデル: {best_model_name}")
print(f"■ 提出ファイル: {SUBMISSION_PATH}")
print("\n■ Leaderboard (上位5モデル):")
print(leaderboard.head())

[2026-08-08 14:51:11] [INFO] predict_proba columns: [0, 1]


INFO:13_autogluon_prototype:predict_proba columns: [0, 1]


[2026-08-08 14:51:11] [INFO] 提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_13_autogluon_prototype_autogluon.csv


INFO:13_autogluon_prototype:提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_13_autogluon_prototype_autogluon.csv



■ AutoGluon 検証スコア (時系列ホールドアウト, Log Loss): 0.567022
■ 使用モデル: WeightedEnsemble_L2
■ 提出ファイル: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_13_autogluon_prototype_autogluon.csv

■ Leaderboard (上位5モデル):
                 model  score_test  score_val eval_metric  pred_time_test  \
0  WeightedEnsemble_L2   -0.567022  -0.567022    log_loss        0.118088   
1             CatBoost   -0.567941  -0.567941    log_loss        0.023682   
2           LightGBMXT   -0.578975  -0.578975    log_loss        0.008906   
3             LightGBM   -0.584119  -0.584119    log_loss        0.007805   
4       ExtraTreesGini   -0.587693  -0.587693    log_loss        0.134421   

   pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  \
0       0.095726  23.699342                 0.004116                0.000705   
1       0.021028  10.994978                 0.023682                0.021028   
2       0.006332   3.993580                 0.008906                0.006332   

## 6b. 比較用: 単体最良モデル（アンサンブルでない）での提出ファイルも作成

12_で判明した「OOFに合わせて重みを学習するアンサンブル（Stacking/Optimized Weighted Average）はOOFスコアこそ良いが、
Publicでは単純な固定ヒューリスティックより悪化する」というパターンが、AutoGluonの`WeightedEnsemble`
（検証データ=`tuning_data`に対してLog Lossを最小化するように重みを学習する仕組み）にも当てはまる可能性がある。

そこで、AutoGluonの既定（Weighted Ensemble）に加えて、**アンサンブルでない単体最良モデル**でも
提出ファイルを作成し、両者のPublicスコアを比較できるようにする。

※ 既に`predictor.fit()`まで実行済みのセッションであれば、このセルだけを一番下に追加して実行すれば
（再学習不要で）動作する。セッションが切れている場合は、次のセルのコメントを参照して
`TabularPredictor.load()`で保存済みモデルを読み込んでから実行すること。

In [20]:
# セッションが切れていて predictor / leaderboard / X_test / positive_col が残っていない場合は、
# 先に以下のコメントアウトを外して保存済みモデルを読み込んでから実行する。
#
# from autogluon.tabular import TabularPredictor
# predictor = TabularPredictor.load(str(AUTOGLUON_MODEL_DIR))
# leaderboard = predictor.leaderboard(ag_tuning_data, silent=True)
# score_col = "score_test" if "score_test" in leaderboard.columns else "score_val"
# best_model_name = predictor.model_best
# best_score = leaderboard.loc[leaderboard["model"] == best_model_name, score_col].values[0]
# X_test = test_features.copy()
# proba_df = predictor.predict_proba(X_test)
# positive_col = 1 if 1 in proba_df.columns else proba_df.columns[-1]

# アンサンブル(WeightedEnsemble)を除いた単体モデルの中で最良のものを特定
single_model_leaderboard = leaderboard[~leaderboard["model"].str.contains("WeightedEnsemble", na=False)]
single_model_leaderboard = single_model_leaderboard.sort_values(score_col, ascending=False)
single_best_model = single_model_leaderboard["model"].iloc[0]
single_best_score = single_model_leaderboard[score_col].iloc[0]

logger.info(f"単体最良モデル: {single_best_model}, Log Loss (時系列ホールドアウト): {-single_best_score:.6f}")
logger.info(f"(参考) Weighted Ensemble: {best_model_name}, Log Loss (時系列ホールドアウト): {-best_score:.6f}")

proba_single_df = predictor.predict_proba(X_test, model=single_best_model)
test_preds_single = proba_single_df[positive_col].values

# ファイル名に使えない文字が model 名に含まれる場合に備えて簡易サニタイズ
safe_model_name = "".join(c if c.isalnum() or c == "_" else "_" for c in single_best_model)
SINGLE_BEST_SUBMISSION_PATH = OUTPUT_DIR / f"{TODAY}_{SCRIPT_NAME}_single_best_{safe_model_name}.csv"

sub_single = pd.DataFrame({ID_COL: X_test.index, TARGET_COL: test_preds_single})
sub_single.to_csv(SINGLE_BEST_SUBMISSION_PATH, index=False, header=False)
logger.info(f"単体最良モデルの提出ファイル保存: {SINGLE_BEST_SUBMISSION_PATH}")

print(f"\n■ Weighted Ensemble  : {best_model_name} (Log Loss {-best_score:.6f}) -> {SUBMISSION_PATH}")
print(f"■ 単体最良モデル      : {single_best_model} (Log Loss {-single_best_score:.6f}) -> {SINGLE_BEST_SUBMISSION_PATH}")
print("\n※ 両方をKaggleに提出し、Publicスコアを比較することで、AutoGluonのWeighted Ensembleが")
print("   12_のStacking/Optimized Weighted Averageと同様にOOFへ過学習していないかを検証する。")

[2026-08-08 15:04:44] [INFO] 単体最良モデル: CatBoost, Log Loss (時系列ホールドアウト): 0.567941


INFO:13_autogluon_prototype:単体最良モデル: CatBoost, Log Loss (時系列ホールドアウト): 0.567941


[2026-08-08 15:04:44] [INFO] (参考) Weighted Ensemble: WeightedEnsemble_L2, Log Loss (時系列ホールドアウト): 0.567022


INFO:13_autogluon_prototype:(参考) Weighted Ensemble: WeightedEnsemble_L2, Log Loss (時系列ホールドアウト): 0.567022


[2026-08-08 15:04:44] [INFO] 単体最良モデルの提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_13_autogluon_prototype_single_best_CatBoost.csv


INFO:13_autogluon_prototype:単体最良モデルの提出ファイル保存: /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_13_autogluon_prototype_single_best_CatBoost.csv



■ Weighted Ensemble  : WeightedEnsemble_L2 (Log Loss 0.567022) -> /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_13_autogluon_prototype_autogluon.csv
■ 単体最良モデル      : CatBoost (Log Loss 0.567941) -> /content/drive/MyDrive/jaggle_2026/data/output/20260808/20260808_13_autogluon_prototype_single_best_CatBoost.csv

※ 両方をKaggleに提出し、Publicスコアを比較することで、AutoGluonのWeighted Ensembleが
   12_のStacking/Optimized Weighted Averageと同様にOOFへ過学習していないかを検証する。


## 7. まとめ・次のアクション

### 12_との比較のポイント
- 11_/12_のOOF Log Loss（0.585〜0.591程度）と、本ノートブックの時系列ホールドアウトLog Lossを比較する。
  ただし検証データの作り方が異なる（12_はTimeSeriesSplit 6-fold平均、本ノートブックは単一の時系列ホールドアウト）ため、
  厳密な比較には注意が必要。
- AutoGluonが個別モデル（LightGBM/CatBoost/XGBoost/NN等）をどう重み付けしたか（`predictor.leaderboard()`の
  `pred_time_val`や`fit_time`列、`predictor.info()`のensemble weightsも参照）を確認すると、
  手動アンサンブル（12_のStacking/Optimized Weighted Average）との違いが見えてくる。

### 次のアクション候補
1. 本プロトタイプが良好であれば、時系列ホールドアウトを複数期間用意した簡易的な時系列CVをAutoGluonに組み込む
   （例: 複数の`tuning_data`で学習を繰り返し予測を平均する等）
2. `time_limit`を伸ばす、`presets='high_quality'`等を試す（ただしAutoGluonの内部バギングが有効になり、
   ランダムK-foldに戻る点に注意）
3. AutoGluonが自動生成したテキスト特徴量・カテゴリ処理を`predictor.feature_importance()`で確認し、
   12_で手動生成した特徴量のうちどれが実際に効いているかを検証する
